# Tahap 7 — Validation
Model final sudah dikunci di Tahap 6. Notebook ini pertama kali membaca data OOT 2025.

In [1]:
import json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from sklearn.inspection import permutation_importance

from src import config
from src.data_prep import load_processed
from src.evaluate import bootstrap_ci, metrics_at
from src.explain import fairness_table, readable_name, reason_codes, segment_metrics, shap_explain
from src.reporting import fmt_id, localize_figure, save_figure, save_table, set_plot_style

set_plot_style()
LBL = config.FEATURE_LABELS
meta = json.loads((config.MODELS / "final_threshold.json").read_text())
model = joblib.load(config.MODELS / "final_model.joblib")
THR = meta["threshold"]
print(meta)

X_train, y_train, s_train, m_train = load_processed("train")
X_test, y_test, s_test, m_test = load_processed("test")
X_oot, y_oot, s_oot, m_oot = load_processed("oot")
p_train = model.predict_proba(X_train)[:, 1]
p_test = model.predict_proba(X_test)[:, 1]
p_oot = model.predict_proba(X_oot)[:, 1]

{'model': 'xgb', 'threshold': 0.17, 'c_fn': 5, 'c_fp': 1, 'calibrated': True, 'calibration_method': 'isotonic'}


## 1. Overfitting / underfitting: train vs CV vs test

In [2]:
cv = pd.read_csv(config.TABLES / "t5_cv_results.csv")
cv = cv[cv["model"] == meta["model"]].set_index("metrik")["mean"]
m_tr, m_te = metrics_at(y_train, p_train, THR), metrics_at(y_test, p_test, THR)
gap = pd.DataFrame([
    {"metrik": "roc_auc", "train": m_tr["roc_auc"], "cv_5_fold": cv["roc_auc"], "test": m_te["roc_auc"]},
    {"metrik": "pr_auc", "train": m_tr["pr_auc"], "cv_5_fold": cv["average_precision"], "test": m_te["pr_auc"]},
    {"metrik": "recall", "train": m_tr["recall"], "cv_5_fold": np.nan, "test": m_te["recall"]},
    {"metrik": "precision", "train": m_tr["precision"], "cv_5_fold": np.nan, "test": m_te["precision"]},
    {"metrik": "cost_per_1000", "train": m_tr["cost_per_1000"], "cv_5_fold": np.nan, "test": m_te["cost_per_1000"]},
])
gap["selisih_train_test"] = gap["train"] - gap["test"]
gap["flag_gap_gt_0_05"] = (gap["metrik"].isin(["roc_auc", "pr_auc"])) & (gap["selisih_train_test"].abs() > 0.05)
save_table(gap, "t7_train_test_gap", decimals=3)
gap

,metrik,train,cv_5_fold,test,selisih_train_test,flag_gap_gt_0_05
0,roc_auc,0.953977,0.907216,0.907003,0.046974,False
1,pr_auc,0.845544,0.796277,0.789935,0.055609,True
2,recall,0.789596,NaN,0.759474,0.030122,False
3,precision,0.728587,NaN,0.692037,0.036550,False
4,cost_per_1000,158.200950,NaN,181.074448,-22.873498,False


Catatan: CV recall/precision Tahap 5 dihitung pada threshold 0,5, sehingga tidak sebanding dengan threshold final; hanya AUC yang dibandingkan.

## 2. Stabilitas: bootstrap 1.000× pada test

In [3]:
boot = bootstrap_ci(y_test, p_test, THR)
save_table(boot, "t7_bootstrap_ci", decimals=3)
boot

,metrik,estimasi,ci_bawah,ci_atas,n_valid
0,roc_auc,0.907003,0.894842,0.917696,1000
1,pr_auc,0.789935,0.769959,0.808734,1000
2,recall,0.759474,0.736759,0.781469,1000


## 3. Robustness lintas waktu: test 2024 vs OOT 2025 (threshold tetap)

In [4]:
oot_rows = []
for name, y, p in (("Test 2024", y_test, p_test), ("OOT 2025", y_oot, p_oot)):
    oot_rows.append({"data": name, "n": len(y), "proporsi_ditolak": y.mean(), **metrics_at(y, p, THR)})
oot_perf = pd.DataFrame(oot_rows)
save_table(oot_perf, "t7_oot_performance", decimals=3)
oot_perf

,data,n,proporsi_ditolak,accuracy,precision,recall,specificity,f1,roc_auc,pr_auc,brier,cost_per_1000
0,Test 2024,11001,0.117535,0.932006,0.692037,0.759474,0.954986,0.724189,0.907003,0.789935,0.044144,181.074448
1,OOT 2025,53241,0.096730,0.930016,0.619263,0.717864,0.952735,0.664928,0.890896,0.728816,0.042896,179.147649


### Sensitivitas: hanya aplikasi berdata lengkap
Bila model masih memanfaatkan pola kosong, kinerjanya pada data lengkap akan jauh lebih rendah daripada pada semua data.

In [5]:
CORE = ["property_value", "income", "dti_mid"]
sens_rows = []
for name, X, y, p in (("Test 2024", X_test, y_test, p_test), ("OOT 2025", X_oot, y_oot, p_oot)):
    complete = X[CORE].notna().all(axis=1).to_numpy()
    for subset, m in (("semua aplikasi", np.ones(len(y), bool)), ("data lengkap", complete), ("ada yang kosong", ~complete)):
        sens_rows.append({"data": name, "subset": subset, "n": int(m.sum()), "proporsi_ditolak": y[m].mean(),
                          **{k: v for k, v in metrics_at(y[m], p[m], THR).items() if k in ("roc_auc", "pr_auc", "recall", "precision", "cost_per_1000")}})
sens = pd.DataFrame(sens_rows)
save_table(sens, "t7_complete_data_sensitivity", decimals=3)
sens

,data,subset,n,proporsi_ditolak,precision,recall,roc_auc,pr_auc,cost_per_1000
0,Test 2024,semua aplikasi,11001,0.117535,0.692037,0.759474,0.907003,0.789935,181.074448
1,Test 2024,data lengkap,10621,0.106770,0.685924,0.743386,0.901170,0.772404,173.335844
2,Test 2024,ada yang kosong,380,0.418421,0.731579,0.874214,0.912519,0.908927,397.368421
3,OOT 2025,semua aplikasi,53241,0.096730,0.619263,0.717864,0.890896,0.728816,179.147649
4,OOT 2025,data lengkap,51600,0.089109,0.607384,0.704872,0.886067,0.717555,172.093023
5,OOT 2025,ada yang kosong,1641,0.336380,0.719243,0.826087,0.887069,0.849905,400.975015


## 4. Robustness segmen (test 2024)

In [6]:
def quartile_segments(col: str) -> pd.Series:
    edges = np.quantile(X_train[col].dropna(), [0, 0.25, 0.5, 0.75, 1])
    edges[0], edges[-1] = -np.inf, np.inf
    labels = [f"Q{i}" for i in range(1, 5)]
    return pd.cut(X_test[col], edges, labels=labels).astype(object)


top_msa = m_train["derived_msa-md"].value_counts().head(5).index
msa = m_test["derived_msa-md"].where(m_test["derived_msa-md"].isin(top_msa))
seg_tables = []
for dim, seg in (("Kuartil jumlah pinjaman", quartile_segments("loan_amount")),
                 ("Kuartil pendapatan", quartile_segments("income")),
                 ("MSA terbesar (kode)", msa)):
    t = segment_metrics(y_test, p_test, THR, seg)
    t.insert(0, "dimensi", dim)
    seg_tables.append(t.sort_values("segmen"))
segments = pd.concat(seg_tables, ignore_index=True)
save_table(segments, "t7_segment_performance", decimals=3)
segments

,dimensi,segmen,n,denial_rate,roc_auc,pr_auc,recall,cost_per_1000
0,Kuartil jumlah pinjaman,Q1,2781,0.307084,0.931609,0.900107,0.880562,292.700467
1,Kuartil jumlah pinjaman,Q2,2707,0.056890,0.844732,0.626619,0.590909,129.663835
2,Kuartil jumlah pinjaman,Q3,2898,0.046239,0.794425,0.474005,0.492537,130.089717
3,Kuartil jumlah pinjaman,Q4,2615,0.057744,0.802420,0.510510,0.483444,172.084130
4,Kuartil pendapatan,Q1,2756,0.285922,0.941041,0.908894,0.880711,257.256894
5,Kuartil pendapatan,Q2,2737,0.079284,0.870049,0.671577,0.686636,155.644867
6,Kuartil pendapatan,Q3,2640,0.050758,0.823072,0.521456,0.522388,137.121212
7,Kuartil pendapatan,Q4,2719,0.042295,0.724701,0.328580,0.313043,163.663111
8,MSA terbesar (kode),17820,1117,0.106535,0.912114,0.770271,0.756303,174.574754
9,MSA terbesar (kode),19740,6040,0.092384,0.895394,0.745272,0.700717,167.052980


## 5. Pemeriksaan leakage

In [7]:
feats = list(X_train.columns)
assert set(feats).isdisjoint(config.LEAKAGE_COLS + config.PROTECTED_COLS + config.PROXY_COLS + config.LEAKAGE_AUDIT_DROPPED)
print("Fitur final:", feats)
if m_te["roc_auc"] > 0.95:
    print(f"RED FLAG: ROC-AUC test {m_te['roc_auc']:.3f} > 0,95 — periksa leakage")
else:
    print(f"ROC-AUC test {m_te['roc_auc']:.3f} ≤ 0,95 — tidak ada red flag AUC")

Fitur final: ['loan_amount', 'income', 'property_value', 'ltv_calc', 'dti_mid', 'loan_term', 'tract_to_msa_income_percentage', 'ffiec_msa_md_median_family_income', 'loan_to_income', 'property_to_income', 'submission_of_application', 'conforming_loan_limit', 'has_coapplicant']
ROC-AUC test 0.907 ≤ 0,95 — tidak ada red flag AUC


## 6. Explainability — SHAP (sampel 5.000 baris test)

In [8]:
sample_idx = X_test.sample(min(config.SHAP_SAMPLE, len(X_test)), random_state=config.RANDOM_STATE).index
Xs, ys = X_test.loc[sample_idx], y_test.loc[sample_idx]
background = X_train.sample(1000, random_state=config.RANDOM_STATE)
expl = shap_explain(model, Xs, background=background)
names = list(expl.feature_names)

display_data = np.array(expl.data, dtype=float)
for j, n in enumerate(names):
    if n in Xs.columns and n in config.NUMERIC_FEATURES:
        display_data[:, j] = Xs[n].to_numpy()
ex = shap.Explanation(values=expl.values, base_values=expl.base_values, data=display_data,
                      feature_names=[readable_name(n) for n in names])

share = np.abs(expl.values).mean(axis=0)
share = pd.Series(share / share.sum(), index=names).sort_values(ascending=False)
print("Kontribusi |SHAP| terbesar:", share.head(5).round(3).to_dict())
if share.iloc[0] > 0.5:
    print("PERINGATAN: satu fitur > 50% total |SHAP| — periksa kemungkinan leakage")


def shap_axes_id(fig):
    for ax in fig.axes:
        if "SHAP" in ax.get_xlabel():
            ax.set_xlabel("Nilai SHAP (dampak terhadap skor risiko penolakan)")
        if ax.get_ylabel() == "Feature value":
            ax.set_ylabel("Nilai fitur")
            ax.set_yticklabels(["Rendah", "Tinggi"])
    return fig


shap.plots.beeswarm(ex, max_display=15, show=False)
save_figure(localize_figure(shap_axes_id(plt.gcf())), "f7_shap_summary")
shap.plots.bar(ex, max_display=15, show=False)
fig = plt.gcf()
fig.axes[0].set_xlabel("Rata-rata |nilai SHAP|")
save_figure(localize_figure(fig), "f7_shap_bar")

Kontribusi |SHAP| terbesar: {'dti_mid': 0.3529999852180481, 'property_value': 0.13300000131130219, 'loan_term': 0.10000000149011612, 'ltv_calc': 0.0820000022649765, 'submission_of_application_1': 0.05700000002980232}


PosixPath('/home/sevima-375/Binus/Exam/Business Intelligence & Analytics/hmda/reports/figures/f7_shap_bar.png')

In [9]:
raw_top = [n for n in share.index if n in config.NUMERIC_FEATURES][:3]
for i, n in enumerate(raw_top, start=1):
    shap.plots.scatter(ex[:, names.index(n)], show=False)
    fig = plt.gcf()
    fig.axes[0].set_xlabel(LBL[n])
    fig.axes[0].set_ylabel("Nilai SHAP")
    save_figure(fig, f"f7_shap_dependence_{i}")
raw_top

['dti_mid', 'property_value', 'loan_term']

In [10]:
ps = model.predict_proba(Xs)[:, 1]
examples = {"disetujui": int(np.argmin(ps)), "borderline": int(np.argmin(np.abs(ps - THR))), "ditolak": int(np.argmax(ps))}
rc_rows = []
for label, i in examples.items():
    shap.plots.waterfall(ex[i], max_display=10, show=False)
    save_figure(localize_figure(plt.gcf()), f"f7_shap_waterfall_{label}")
    rc_rows.append({"contoh": label, "probabilitas": ps[i], "aktual": "ditolak" if ys.iloc[i] == 1 else "disetujui",
                    "reason_codes": "; ".join(reason_codes(expl.values[i], names, data=np.asarray(expl.data[i]))) or "-"})
rc = pd.DataFrame(rc_rows)
save_table(rc, "t7_reason_codes_examples", decimals=3)
rc

,contoh,probabilitas,aktual,reason_codes
0,disetujui,0.000000,disetujui,Rasio pinjaman terhadap pendapatan; Pengajuan ...
1,borderline,0.148649,disetujui,Rasio pinjaman terhadap nilai properti (%); Ni...
2,ditolak,1.000000,ditolak,Rasio utang terhadap pendapatan (%); Tenor pin...


## 7. Permutation importance (cross-check)

In [11]:
perm = permutation_importance(model, Xs, ys, scoring="average_precision", n_repeats=5,
                              random_state=config.RANDOM_STATE, n_jobs=config.N_JOBS)
perm_tbl = pd.DataFrame({"fitur": Xs.columns, "penurunan_pr_auc_mean": perm.importances_mean,
                         "penurunan_pr_auc_std": perm.importances_std}).sort_values("penurunan_pr_auc_mean", ascending=False)

raw_share = {}
for n, v in share.items():
    base = n.removeprefix("missingindicator_")
    base = next((c for c in config.CATEGORICAL_FEATURES if base.startswith(c + "_")), base)
    raw_share[base] = raw_share.get(base, 0) + v
raw_share = pd.Series(raw_share)
perm_tbl["porsi_shap"] = perm_tbl["fitur"].map(raw_share)
perm_tbl["peringkat_permutation"] = range(1, len(perm_tbl) + 1)
perm_tbl["peringkat_shap"] = perm_tbl["porsi_shap"].rank(ascending=False, method="min").astype(int)
perm_tbl["label"] = perm_tbl["fitur"].map(LBL)
save_table(perm_tbl, "t7_importance_comparison", decimals=4)

fig, ax = plt.subplots(figsize=(7, 4.5))
pt = perm_tbl.iloc[::-1]
ax.barh(pt["label"], pt["penurunan_pr_auc_mean"], xerr=pt["penurunan_pr_auc_std"], color="#4C72B0")
ax.set_xlabel("Penurunan PR-AUC saat fitur diacak")
save_figure(fig, "f7_perm_importance")
perm_tbl

,fitur,penurunan_pr_auc_mean,penurunan_pr_auc_std,porsi_shap,peringkat_permutation,peringkat_shap,label
4,dti_mid,0.396551,0.015817,0.352576,1,1,Rasio utang terhadap pendapatan (%)
3,ltv_calc,0.032613,0.004751,0.081765,2,4,Rasio pinjaman terhadap nilai properti (%)
2,property_value,0.032260,0.005143,0.133225,3,2,Nilai properti (USD)
5,loan_term,0.025477,0.004859,0.100193,4,3,Tenor pinjaman (bulan)
8,loan_to_income,0.014378,0.001398,0.029853,5,9,Rasio pinjaman terhadap pendapatan
6,tract_to_msa_income_percentage,0.012006,0.000756,0.048973,6,7,Pendapatan tract terhadap MSA (%)
10,submission_of_application,0.009042,0.002399,0.072875,7,5,Pengajuan langsung ke lender
0,loan_amount,0.007724,0.000680,0.053197,8,6,Jumlah pinjaman (USD)
1,income,0.005529,0.000225,0.029708,9,10,Pendapatan (ribu USD)
9,property_to_income,0.004506,0.000404,0.047237,10,8,Rasio nilai properti terhadap pendapatan


## 8. Business sense check

In [12]:
EXPECT = {
    "dti_mid": ("+", "DTI tinggi = kapasitas bayar rendah → risiko naik"),
    "ltv_calc": ("+", "LTV tinggi = uang muka kecil, agunan tipis → risiko naik"),
    "income": ("-", "Pendapatan tinggi → kapasitas bayar naik → risiko turun"),
    "loan_to_income": ("+", "Pinjaman besar relatif pendapatan → risiko naik"),
    "property_to_income": ("+", "Properti mahal relatif pendapatan → beban naik"),
    "loan_amount": ("?", "Efek bergantung pendapatan/agunan; tidak ada ekspektasi tunggal"),
    "property_value": ("?", "Efek bergantung pinjaman/pendapatan"),
    "loan_term": ("?", "Tenor standar 30 tahun dominan; tenor tidak standar bisa berisiko"),
    "tract_to_msa_income_percentage": ("-", "Wilayah berpendapatan lebih tinggi → risiko agunan/ekonomi lebih rendah"),
    "ffiec_msa_md_median_family_income": ("?", "Konteks pasar wilayah; tidak ada ekspektasi tunggal"),
}
bs_rows = []
for n in share.index:
    if n not in EXPECT or n not in Xs.columns:
        continue
    x = Xs[n].to_numpy()
    v = expl.values[:, names.index(n)]
    ok = ~np.isnan(x)
    rho = pd.Series(x[ok]).corr(pd.Series(v[ok]), method="spearman")
    arah = "+" if rho > 0.1 else ("-" if rho < -0.1 else "datar/non-monoton")
    exp_sign, reason = EXPECT[n]
    bs_rows.append({"fitur": LBL[n], "porsi_shap": share[n], "arah_pengaruh": arah, "korelasi_spearman_nilai_vs_shap": rho,
                    "ekspektasi": exp_sign, "logika_underwriting": reason,
                    "sesuai": "-" if exp_sign == "?" else ("ya" if arah == exp_sign else "tidak")})
for col in ("has_coapplicant", "conforming_loan_limit", "submission_of_application"):
    cols = [n for n in names if n.startswith(col + "_")]
    effects = {}
    for c in cols:
        j = names.index(c)
        active = np.asarray(expl.data[:, j]) > 0
        if active.any():
            effects[readable_name(c)] = float(expl.values[active, j].mean())
    bs_rows.append({"fitur": LBL[col], "porsi_shap": raw_share.get(col, np.nan), "arah_pengaruh": "; ".join(f"{k}: {fmt_id(v, 3)}" for k, v in effects.items()),
                    "korelasi_spearman_nilai_vs_shap": np.nan,
                    "ekspektasi": "-" if col == "has_coapplicant" else "?",
                    "logika_underwriting": {"has_coapplicant": "Ada co-applicant menambah pendapatan rumah tangga → risiko turun",
                                            "conforming_loan_limit": "Kredit non-conforming (jumbo) punya syarat lebih ketat",
                                            "submission_of_application": "Jalur pengajuan (langsung/broker) dapat memengaruhi kelengkapan berkas"}[col],
                    "sesuai": "lihat arah per kategori"})
business = pd.DataFrame(bs_rows)
save_table(business, "t7_business_sense", decimals=3)
business

,fitur,porsi_shap,arah_pengaruh,korelasi_spearman_nilai_vs_shap,ekspektasi,logika_underwriting,sesuai
0,Rasio utang terhadap pendapatan (%),0.352576,+,0.627992,+,DTI tinggi = kapasitas bayar rendah → risiko naik,ya
1,Nilai properti (USD),0.133225,-,-0.855954,?,Efek bergantung pinjaman/pendapatan,-
2,Tenor pinjaman (bulan),0.100193,-,-0.447015,?,Tenor standar 30 tahun dominan; tenor tidak st...,-
3,Rasio pinjaman terhadap nilai properti (%),0.081765,+,0.882830,+,"LTV tinggi = uang muka kecil, agunan tipis → r...",ya
4,Jumlah pinjaman (USD),0.053197,-,-0.340139,?,Efek bergantung pendapatan/agunan; tidak ada e...,-
5,Pendapatan tract terhadap MSA (%),0.048973,-,-0.617364,-,Wilayah berpendapatan lebih tinggi → risiko ag...,ya
6,Rasio nilai properti terhadap pendapatan,0.047237,-,-0.840966,+,Properti mahal relatif pendapatan → beban naik,tidak
7,Rasio pinjaman terhadap pendapatan,0.029853,+,0.105849,+,Pinjaman besar relatif pendapatan → risiko naik,ya
8,Pendapatan (ribu USD),0.029708,-,-0.222461,-,Pendapatan tinggi → kapasitas bayar naik → ris...,ya
9,Median pendapatan keluarga MSA (USD),0.020815,-,-0.228537,?,Konteks pasar wilayah; tidak ada ekspektasi tu...,-


## 9. Fairness (atribut protected hanya untuk evaluasi)

In [13]:
pred_test = (p_test >= THR).astype(int)
fair_tables, na_rows = [], []
for col in config.FAIRNESS_COLS:
    t = fairness_table(y_test, pred_test, s_test[col])
    t.insert(0, "atribut", col)
    fair_tables.append(t)
    na = s_test[col].astype(str).str.contains("Not Available|8888|Free Form", regex=True) | s_test[col].isna()
    na_rows.append({"atribut": col, "pct_tidak_tersedia": na.mean() * 100})
fair = pd.concat(fair_tables, ignore_index=True)
save_table(fair, "t7_fairness", decimals=3)
save_table(pd.DataFrame(na_rows), "t7_fairness_missing", decimals=2)
fair

,atribut,grup,n,approval_rate_pred,approval_rate_aktual,tpr,fpr,dir,dir_aktual,eod,catatan
0,derived_race,White,7984,0.891032,0.902305,0.730769,0.041644,0.962916,0.980618,-0.020889,
1,derived_race,Race Not Available,1730,0.778613,0.796532,0.846591,0.061684,0.841428,0.865665,-0.040929,tidak tersedia
2,derived_race,Asian,576,0.925347,0.920139,0.695652,0.020755,1.000000,1.000000,0.000000,referensi
3,derived_race,Joint,371,0.892183,0.919137,0.733333,0.052786,0.964161,0.998912,-0.032031,
4,derived_race,Black or African American,224,0.821429,0.812500,0.642857,0.071429,0.887698,0.883019,-0.050674,
5,derived_race,American Indian or Alaska Native,89,0.640449,0.629213,0.727273,0.142857,0.692118,0.683824,-0.122102,n kecil
6,derived_race,2 or more minority races,19,0.526316,0.578947,0.875000,0.181818,0.568777,0.629196,-0.161063,n kecil
7,derived_race,Native Hawaiian or Other Pacific Islander,8,0.750000,0.750000,1.000000,0.000000,0.810507,0.815094,0.020755,n kecil
8,derived_ethnicity,Not Hispanic or Latino,7572,0.926836,0.927364,0.625455,0.029906,1.000000,1.000000,0.000000,referensi
9,derived_ethnicity,Hispanic or Latino,1529,0.607587,0.670373,0.898810,0.143415,0.655549,0.722880,-0.113509,"DIR < 0,8"
